
# Exploring Model and Brain Representations

This notebook explores representational similarity between:

- a **standard ImageNet-pretrained AlexNet**,
- a **mixed clear/blur fine-tuned AlexNet expert**, and
- the **human inferior temporal (IT) fMRI RDMs** from the Cichy 92-image dataset.

The notebook is intentionally organized for interactive exploration. Choose one model layer in the configuration cell, then rerun the cells below to:

1. load the selected model RDMs and subject-level IT RDMs,
2. inspect the standard, expert, and mean IT RDMs,
3. visualize the subtle expert-minus-standard change,
4. inspect paired subject-level RSA values, and
5. visualize the paired statistical effect.

The primary comparison currently uses **FC7 post-ReLU**, rather than the final category-logit layer.



## 1. Imports and plotting defaults

The figures use `constrained_layout=True` and dedicated colorbar axes to prevent titles, labels, and colorbars from overlapping.


In [ ]:

from __future__ import annotations

from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io as sio
from IPython.display import display
from scipy.stats import spearmanr

plt.rcParams.update(
    {
        "font.size": 11,
        "axes.titlesize": 12,
        "axes.labelsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 10,
        "figure.titlesize": 14,
    }
)



## 2. Configuration

Edit only this cell when changing layers, paths, or output settings.

`SELECTED_LAYER` must match the layer name used in the saved RDM filename:

```text
rdm_<SELECTED_LAYER>_correlation.npy
```

Examples:

- `fc7_pre_relu`
- `fc7_post_relu`
- `fc8_logits`


In [ ]:

# Project paths
PROJECT_ROOT = Path("/scratch/americanod94/lost_glasses")
HOME_PROJECT_ROOT = Path("/home/americanod94/projects/lost_glasses")

CICHY_DATA_DIR = PROJECT_ROOT / "data" / "92_Image_Set"
FMRI_FILE = CICHY_DATA_DIR / "target_fmri.mat"

RDM_ROOT = (
    HOME_PROJECT_ROOT
    / "outputs"
    / "rdms"
    / "cichy_data_models"
)
RSA_DIR = RDM_ROOT / "rsa_analysis"

SUBJECT_RSA_FILE = RSA_DIR / "subject_rsa_values.csv"
PAIRED_COMPARISONS_FILE = RSA_DIR / "paired_model_comparisons.csv"

# Single-layer comparison
SELECTED_LAYER = "fc7_post_relu"

LAYER_DISPLAY_NAMES = {
    "fc7_pre_relu": "FC7 pre-ReLU",
    "fc7_post_relu": "FC7 post-ReLU",
    "fc8_logits": "FC8 logits",
}

MODEL_NAMES = ["standard", "expert"]
MODEL_DISPLAY_NAMES = {
    "standard": "Standard AlexNet",
    "expert": "Mixed-training expert",
}

EXPECTED_STIMULI = 92
SAVE_FIGURES = True
FIGURE_DPI = 300
FIGURE_DIR = RSA_DIR / "figures" / SELECTED_LAYER

print("Selected layer:", SELECTED_LAYER)
print("RDM root:", RDM_ROOT)
print("Figure directory:", FIGURE_DIR)



## 3. Check available model RDMs

This cell discovers the layers currently available for both models. It is useful before changing `SELECTED_LAYER`.


In [ ]:

def discover_available_rdms(rdm_root: Path) -> pd.DataFrame:
    rows = []

    for model_name in MODEL_NAMES:
        model_dir = rdm_root / model_name

        if not model_dir.exists():
            continue

        for rdm_path in sorted(model_dir.glob("rdm_*_correlation.npy")):
            layer_name = rdm_path.name.removeprefix("rdm_").removesuffix(
                "_correlation.npy"
            )

            rows.append(
                {
                    "model": model_name,
                    "layer": layer_name,
                    "path": str(rdm_path),
                }
            )

    return pd.DataFrame(rows)


available_rdms = discover_available_rdms(RDM_ROOT)
display(available_rdms)

available_layers = sorted(available_rdms["layer"].unique())

if SELECTED_LAYER not in available_layers:
    raise ValueError(
        f"Selected layer {SELECTED_LAYER!r} was not found. "
        f"Available layers: {available_layers}"
    )


## 4. Data-loading and validation helpers

In [ ]:

def load_mat_file(mat_path: Path) -> dict[str, np.ndarray]:
    """Load MATLAB v7 files or HDF5-based MATLAB v7.3 files."""

    if not mat_path.exists():
        raise FileNotFoundError(f"MATLAB file not found:\n{mat_path}")

    try:
        with h5py.File(mat_path, "r") as file:
            return {
                name: np.transpose(np.asarray(file[name]))
                for name in file.keys()
            }
    except OSError:
        return sio.loadmat(mat_path)


def standardize_subject_rdm_axes(
    subject_rdms: np.ndarray,
    expected_stimuli: int = EXPECTED_STIMULI,
) -> np.ndarray:
    """Return IT RDMs in [subjects, stimuli, stimuli] order."""

    subject_rdms = np.asarray(subject_rdms, dtype=np.float64)
    expected_pair = (expected_stimuli, expected_stimuli)

    if subject_rdms.ndim != 3:
        raise ValueError(
            f"IT_RDMs must be three-dimensional; got {subject_rdms.shape}."
        )

    if subject_rdms.shape[1:] == expected_pair:
        return subject_rdms

    if subject_rdms.shape[:2] == expected_pair:
        return np.moveaxis(subject_rdms, source=2, destination=0)

    raise ValueError(
        "Could not infer IT_RDMs axis order. "
        f"Received {subject_rdms.shape}."
    )


def load_it_rdms(fmri_file: Path) -> tuple[np.ndarray, np.ndarray]:
    """Load subject-level IT RDMs and their subject mean."""

    data = load_mat_file(fmri_file)

    if "IT_RDMs" not in data:
        raise KeyError(
            "Could not find 'IT_RDMs'. "
            f"Available keys: {list(data.keys())}"
        )

    subject_rdms = standardize_subject_rdm_axes(data["IT_RDMs"])

    if not np.isfinite(subject_rdms).all():
        raise ValueError("IT RDMs contain NaN or infinite values.")

    return subject_rdms, subject_rdms.mean(axis=0)


def load_model_rdm(
    model_name: str,
    layer_name: str,
    rdm_root: Path = RDM_ROOT,
) -> np.ndarray:
    """Load and validate one model RDM."""

    rdm_path = (
        rdm_root
        / model_name
        / f"rdm_{layer_name}_correlation.npy"
    )

    if not rdm_path.exists():
        raise FileNotFoundError(f"Model RDM not found:\n{rdm_path}")

    rdm = np.load(rdm_path).astype(np.float64, copy=False)
    expected_shape = (EXPECTED_STIMULI, EXPECTED_STIMULI)

    if rdm.shape != expected_shape:
        raise ValueError(
            f"Expected {expected_shape}, but {rdm_path} has {rdm.shape}."
        )

    if not np.isfinite(rdm).all():
        raise ValueError(f"{rdm_path} contains non-finite values.")

    if not np.allclose(rdm, rdm.T, atol=1e-5):
        raise ValueError(f"{rdm_path} is not symmetric.")

    return rdm


def load_rsa_tables() -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load the subject-level RSA values and paired test output."""

    if not SUBJECT_RSA_FILE.exists():
        raise FileNotFoundError(
            f"Subject-level RSA CSV not found:\n{SUBJECT_RSA_FILE}"
        )

    if not PAIRED_COMPARISONS_FILE.exists():
        raise FileNotFoundError(
            f"Paired-comparison CSV not found:\n{PAIRED_COMPARISONS_FILE}"
        )

    subject_rsa = pd.read_csv(SUBJECT_RSA_FILE)
    paired_results = pd.read_csv(PAIRED_COMPARISONS_FILE)

    required = {"subject", "model", "layer", "rho"}
    missing = required - set(subject_rsa.columns)

    if missing:
        raise ValueError(
            f"subject_rsa_values.csv is missing: {sorted(missing)}"
        )

    return subject_rsa, paired_results


def vectorize_rdm(rdm: np.ndarray) -> np.ndarray:
    """Return the lower triangle without the diagonal."""

    indices = np.tril_indices(rdm.shape[0], k=-1)
    return rdm[indices]


def bootstrap_mean_ci(
    values: np.ndarray,
    n_bootstrap: int = 20_000,
    confidence: float = 0.95,
    seed: int = 42,
) -> tuple[float, float]:
    """Percentile bootstrap confidence interval for a mean."""

    values = np.asarray(values, dtype=np.float64)
    rng = np.random.default_rng(seed)

    sample_indices = rng.integers(
        low=0,
        high=len(values),
        size=(n_bootstrap, len(values)),
    )

    bootstrap_means = values[sample_indices].mean(axis=1)
    tail = (1.0 - confidence) / 2.0

    return (
        float(np.quantile(bootstrap_means, tail)),
        float(np.quantile(bootstrap_means, 1.0 - tail)),
    )


def robust_symmetric_limit(
    matrix: np.ndarray,
    percentile: float = 99.0,
) -> float:
    """Robust symmetric limit for a zero-centered heatmap."""

    values = np.abs(matrix[np.isfinite(matrix)])

    if values.size == 0:
        raise ValueError("No finite values available.")

    limit = float(np.percentile(values, percentile))

    if limit == 0.0:
        limit = float(values.max())

    return limit if limit > 0.0 else 1e-6


def significance_label(p_value: float) -> str:
    if p_value < 0.001:
        return "***"
    if p_value < 0.01:
        return "**"
    if p_value < 0.05:
        return "*"
    return "n.s."


def save_figure(
    figure: plt.Figure,
    filename: str,
) -> None:
    """Optionally save PNG and PDF versions."""

    if not SAVE_FIGURES:
        return

    FIGURE_DIR.mkdir(parents=True, exist_ok=True)

    figure.savefig(
        FIGURE_DIR / f"{filename}.png",
        dpi=FIGURE_DPI,
        bbox_inches="tight",
    )
    figure.savefig(
        FIGURE_DIR / f"{filename}.pdf",
        bbox_inches="tight",
    )



## 5. Load the selected layer

The cells below are layer-agnostic. Change `SELECTED_LAYER` above and rerun from this point onward.


In [ ]:

subject_it_rdms, mean_it_rdm = load_it_rdms(FMRI_FILE)

standard_rdm = load_model_rdm(
    model_name="standard",
    layer_name=SELECTED_LAYER,
)

expert_rdm = load_model_rdm(
    model_name="expert",
    layer_name=SELECTED_LAYER,
)

subject_rsa, paired_results = load_rsa_tables()

selected_subject_rsa = subject_rsa[
    subject_rsa["layer"] == SELECTED_LAYER
].copy()

selected_paired_results = paired_results[
    paired_results["layer"] == SELECTED_LAYER
].copy()

if selected_subject_rsa.empty:
    raise ValueError(
        f"No subject-level RSA values found for {SELECTED_LAYER}."
    )

if len(selected_paired_results) != 1:
    raise ValueError(
        f"Expected one paired comparison for {SELECTED_LAYER}, "
        f"found {len(selected_paired_results)}."
    )

layer_title = LAYER_DISPLAY_NAMES.get(
    SELECTED_LAYER,
    SELECTED_LAYER,
)

print("IT RDMs:", subject_it_rdms.shape)
print("Standard model RDM:", standard_rdm.shape)
print("Expert model RDM:", expert_rdm.shape)
print("Subject-level RSA rows:", len(selected_subject_rsa))


## 6. Numerical summary

In [ ]:

summary_table = (
    selected_subject_rsa
    .groupby("model")["rho"]
    .agg(["count", "mean", "std", "median"])
    .reindex(MODEL_NAMES)
    .reset_index()
)

display(summary_table)
display(selected_paired_results)



## 7. Standard, expert, and mean IT RDMs

The two model RDMs use a shared color scale, allowing direct comparison. The IT RDM has its own scale because model correlation distances and fMRI dissimilarities need not have the same numerical range.


In [ ]:

model_vmin = min(float(standard_rdm.min()), float(expert_rdm.min()))
model_vmax = max(float(standard_rdm.max()), float(expert_rdm.max()))

figure = plt.figure(
    figsize=(14.2, 4.4),
    constrained_layout=True,
)

grid = figure.add_gridspec(
    1,
    5,
    width_ratios=[1.0, 1.0, 0.045, 1.0, 0.045],
    wspace=0.10,
)

standard_axis = figure.add_subplot(grid[0, 0])
expert_axis = figure.add_subplot(grid[0, 1])
model_cbar_axis = figure.add_subplot(grid[0, 2])
brain_axis = figure.add_subplot(grid[0, 3])
brain_cbar_axis = figure.add_subplot(grid[0, 4])

standard_image = standard_axis.imshow(
    standard_rdm,
    aspect="equal",
    interpolation="nearest",
    cmap="viridis",
    vmin=model_vmin,
    vmax=model_vmax,
)

expert_axis.imshow(
    expert_rdm,
    aspect="equal",
    interpolation="nearest",
    cmap="viridis",
    vmin=model_vmin,
    vmax=model_vmax,
)

brain_image = brain_axis.imshow(
    mean_it_rdm,
    aspect="equal",
    interpolation="nearest",
    cmap="viridis",
)

standard_axis.set_title(
    f"Standard AlexNet\n{layer_title}",
    pad=10,
)
expert_axis.set_title(
    f"Mixed-training expert\n{layer_title}",
    pad=10,
)
brain_axis.set_title(
    "Mean human IT\nfMRI RDM",
    pad=10,
)

for axis in (standard_axis, expert_axis, brain_axis):
    axis.set_xlabel("Stimulus")
    axis.set_xticks([0, 45, 91])
    axis.set_yticks([0, 45, 91])

standard_axis.set_ylabel("Stimulus")
expert_axis.set_yticklabels([])
brain_axis.set_yticklabels([])

model_colorbar = figure.colorbar(
    standard_image,
    cax=model_cbar_axis,
)
model_colorbar.set_label(
    "Correlation distance",
    labelpad=8,
)

brain_colorbar = figure.colorbar(
    brain_image,
    cax=brain_cbar_axis,
)
brain_colorbar.set_label(
    "fMRI dissimilarity",
    labelpad=8,
)

save_figure(
    figure,
    f"{SELECTED_LAYER}_rdm_standard_expert_it",
)

plt.show()



## 8. Expert-minus-standard difference RDM

This heatmap isolates the subtle representational change produced by mixed clear/blur training:

- **positive values**: the expert places the stimulus pair farther apart;
- **negative values**: the expert places the stimulus pair closer together;
- **zero**: no change.

The diverging color scale is centered on zero. The default 99th-percentile limit prevents a small number of extreme cells from hiding the broader pattern.


In [ ]:

difference_rdm = expert_rdm - standard_rdm
difference_limit = robust_symmetric_limit(
    difference_rdm,
    percentile=99.0,
)

mean_absolute_change = float(np.mean(np.abs(difference_rdm)))
median_absolute_change = float(np.median(np.abs(difference_rdm)))
maximum_absolute_change = float(np.max(np.abs(difference_rdm)))

print(f"Mean absolute change:   {mean_absolute_change:.6f}")
print(f"Median absolute change: {median_absolute_change:.6f}")
print(f"Maximum absolute change:{maximum_absolute_change:.6f}")

figure = plt.figure(
    figsize=(7.5, 6.2),
    constrained_layout=True,
)

grid = figure.add_gridspec(
    1,
    2,
    width_ratios=[1.0, 0.045],
    wspace=0.08,
)

axis = figure.add_subplot(grid[0, 0])
colorbar_axis = figure.add_subplot(grid[0, 1])

difference_image = axis.imshow(
    difference_rdm,
    aspect="equal",
    interpolation="nearest",
    cmap="coolwarm",
    vmin=-difference_limit,
    vmax=difference_limit,
)

axis.set_title(
    f"Change in {layer_title} geometry\n"
    "Mixed-training expert − standard AlexNet",
    pad=12,
)
axis.set_xlabel("Stimulus")
axis.set_ylabel("Stimulus")
axis.set_xticks([0, 45, 91])
axis.set_yticks([0, 45, 91])

colorbar = figure.colorbar(
    difference_image,
    cax=colorbar_axis,
)
colorbar.set_label(
    "Change in correlation distance",
    labelpad=8,
)

save_figure(
    figure,
    f"{SELECTED_LAYER}_rdm_expert_minus_standard",
)

plt.show()



## 9. Paired subject-level model–IT correspondence

Each line connects the standard and expert RSA correlations for the same fMRI participant. Diamonds show group means with bootstrap 95% confidence intervals.


In [ ]:

pivot = (
    selected_subject_rsa
    .pivot(index="subject", columns="model", values="rho")
    .reindex(columns=MODEL_NAMES)
    .dropna()
)

standard_values = pivot["standard"].to_numpy()
expert_values = pivot["expert"].to_numpy()

paired_row = selected_paired_results.iloc[0]
p_fdr = float(paired_row["permutation_p_fdr"])

figure, axis = plt.subplots(
    figsize=(6.6, 6.2),
    constrained_layout=True,
)

for _, row in pivot.iterrows():
    axis.plot(
        [0, 1],
        [row["standard"], row["expert"]],
        marker="o",
        markersize=4.5,
        linewidth=0.9,
        alpha=0.50,
        zorder=1,
    )

for position, values in (
    (0, standard_values),
    (1, expert_values),
):
    mean_value = float(values.mean())
    ci_low, ci_high = bootstrap_mean_ci(values)

    axis.errorbar(
        position,
        mean_value,
        yerr=[
            [mean_value - ci_low],
            [ci_high - mean_value],
        ],
        fmt="D",
        markersize=8,
        capsize=5,
        linewidth=2.2,
        zorder=4,
    )

all_values = np.concatenate([standard_values, expert_values])
data_min = float(all_values.min())
data_max = float(all_values.max())
data_range = max(data_max - data_min, 0.05)

bracket_y = data_max + 0.12 * data_range
bracket_height = 0.035 * data_range

axis.plot(
    [0, 0, 1, 1],
    [
        bracket_y,
        bracket_y + bracket_height,
        bracket_y + bracket_height,
        bracket_y,
    ],
    linewidth=1.2,
    clip_on=False,
)

axis.text(
    0.5,
    bracket_y + bracket_height + 0.015 * data_range,
    f"{significance_label(p_fdr)}   $p_{{FDR}}$ = {p_fdr:.4g}",
    ha="center",
    va="bottom",
)

axis.set_ylim(
    data_min - 0.10 * data_range,
    bracket_y + 0.20 * data_range,
)
axis.set_xlim(-0.35, 1.35)
axis.set_xticks(
    [0, 1],
    ["Standard\nAlexNet", "Mixed-training\nexpert"],
)
axis.set_ylabel("Spearman correlation with IT RDM")
axis.set_title(
    f"{layer_title} correspondence with human IT",
    pad=12,
)
axis.axhline(
    0.0,
    linestyle="--",
    linewidth=0.8,
    alpha=0.55,
    zorder=0,
)

save_figure(
    figure,
    f"{SELECTED_LAYER}_paired_subject_rsa",
)

plt.show()



## 10. Paired expert-minus-standard RSA differences

This plot shows the exact subject-level quantity tested by the paired comparison:

```text
expert RSA − standard RSA
```

Values below zero indicate stronger IT correspondence for the standard model. Values above zero indicate stronger IT correspondence for the mixed-training expert.


In [ ]:

differences = expert_values - standard_values
mean_difference = float(differences.mean())
ci_low, ci_high = bootstrap_mean_ci(differences)

print(f"Mean expert − standard difference: {mean_difference:.6f}")
print(f"Bootstrap 95% CI: [{ci_low:.6f}, {ci_high:.6f}]")
print(f"FDR-corrected permutation p: {p_fdr:.6g}")

figure, axis = plt.subplots(
    figsize=(5.8, 6.1),
    constrained_layout=True,
)

axis.boxplot(
    [differences],
    positions=[0],
    widths=0.30,
    showfliers=False,
    medianprops={"linewidth": 2.0},
)

rng = np.random.default_rng(42)
jitter = rng.normal(
    loc=-0.06,
    scale=0.035,
    size=len(differences),
)

axis.scatter(
    jitter,
    differences,
    alpha=0.78,
    s=35,
    zorder=3,
)

axis.errorbar(
    0.14,
    mean_difference,
    yerr=[
        [mean_difference - ci_low],
        [ci_high - mean_difference],
    ],
    fmt="D",
    markersize=8,
    capsize=5,
    linewidth=2.2,
    zorder=4,
    label="Mean and 95% bootstrap CI",
)

axis.axhline(
    0.0,
    linestyle="--",
    linewidth=1.0,
)

y_min = min(float(differences.min()), ci_low)
y_max = max(float(differences.max()), ci_high)
y_range = max(y_max - y_min, 0.005)

axis.set_ylim(
    y_min - 0.12 * y_range,
    y_max + 0.35 * y_range,
)

axis.text(
    0,
    y_max + 0.11 * y_range,
    (
        f"Mean = {mean_difference:.4f}\n"
        f"95% CI [{ci_low:.4f}, {ci_high:.4f}]\n"
        f"$p_{{FDR}}$ = {p_fdr:.4g}"
    ),
    ha="center",
    va="bottom",
)

axis.set_xlim(-0.42, 0.42)
axis.set_xticks([0], [layer_title])
axis.set_ylabel(
    "RSA difference\n"
    "mixed expert − standard"
)
axis.set_title(
    "Effect of mixed clear/blur training\n"
    "on IT correspondence",
    pad=12,
)
axis.legend(
    frameon=False,
    loc="lower right",
)

save_figure(
    figure,
    f"{SELECTED_LAYER}_expert_minus_standard_rsa",
)

plt.show()



## 11. Optional: model RDM correspondence with the mean IT RDM

This cell reports the Spearman correlation between each model RDM and the subject-averaged IT RDM. The subject-level paired analysis above remains the appropriate basis for inferential statistics.


In [ ]:

mean_it_vector = vectorize_rdm(mean_it_rdm)

mean_it_results = []

for model_name, model_rdm in (
    ("standard", standard_rdm),
    ("expert", expert_rdm),
):
    result = spearmanr(
        vectorize_rdm(model_rdm),
        mean_it_vector,
    )

    mean_it_results.append(
        {
            "model": MODEL_DISPLAY_NAMES[model_name],
            "layer": layer_title,
            "spearman_rho": float(result.statistic),
            "p_value": float(result.pvalue),
        }
    )

display(pd.DataFrame(mean_it_results))



## 12. Optional: inspect the largest representational changes

This table identifies stimulus pairs whose correlation distance changed the most between the standard and mixed-training models. The row and column indices refer to the saved `stimulus_order.txt`.


In [ ]:

number_of_pairs_to_show = 20

lower_indices = np.tril_indices(EXPECTED_STIMULI, k=-1)

pair_changes = pd.DataFrame(
    {
        "stimulus_i": lower_indices[0],
        "stimulus_j": lower_indices[1],
        "standard_distance": standard_rdm[lower_indices],
        "expert_distance": expert_rdm[lower_indices],
        "expert_minus_standard": difference_rdm[lower_indices],
        "absolute_change": np.abs(difference_rdm[lower_indices]),
    }
)

largest_changes = (
    pair_changes
    .sort_values("absolute_change", ascending=False)
    .head(number_of_pairs_to_show)
    .reset_index(drop=True)
)

display(largest_changes)



## Notes for interpretation

- A visible difference in the expert-minus-standard RDM does not by itself imply improved brain correspondence.
- The paired subject-level RSA plot and paired-difference plot show whether the representational change consistently increases or decreases correspondence with IT.
- With `fc7_post_relu`, the current results indicate a small but consistent advantage for the standard model.
- The original Cichy images are used here. A separate experiment would be needed to compare clear and synthetically blurred versions of those same stimuli.
